<a href="https://colab.research.google.com/github/impericalskibidi/Titanic-Dataset-/blob/main/Projectlast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Customer Churn Prediction Classifier
=====================================
Binary classification using Logistic Regression and Decision Trees
with comprehensive evaluation, confusion matrix dashboards, and precision-recall analysis.

Usage:
    python churn_classification.py

Output Files Generated:
    - confusion_matrix_dashboard.png
    - precision_recall_curves.png
    - roc_curves.png
    - model_comparison.png
    - churn_classification_report.txt
    - interpretation_guide.txt
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    precision_recall_curve, roc_curve, auc
)
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')


class ChurnClassifier:
    """Customer churn classification pipeline"""

    def __init__(self, random_state=42):
        self.random_state = random_state
        self.df = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.X_train_scaled = None
        self.X_test_scaled = None
        self.lr_model = None
        self.dt_model = None
        self.metrics_lr = None
        self.metrics_dt = None

    def generate_dataset(self, n_samples=1000):
        """Generate synthetic customer churn dataset"""
        print("Generating dataset...")
        np.random.seed(self.random_state)

        data = {
            'CustomerID': range(1, n_samples + 1),
            'Age': np.random.randint(18, 80, n_samples),
            'Tenure_Months': np.random.randint(1, 72, n_samples),
            'Monthly_Charge': np.random.uniform(20, 150, n_samples),
            'Total_Charges': np.random.uniform(100, 8000, n_samples),
            'Internet_Service': np.random.choice(['DSL', 'Fiber', 'No'], n_samples, p=[0.4, 0.4, 0.2]),
            'Contract_Type': np.random.choice(['Month-to-Month', '1 Year', '2 Year'], n_samples, p=[0.4, 0.3, 0.3]),
            'Tech_Support': np.random.choice(['Yes', 'No'], n_samples, p=[0.3, 0.7]),
            'Online_Security': np.random.choice(['Yes', 'No'], n_samples, p=[0.3, 0.7]),
        }

        self.df = pd.DataFrame(data)

        # Generate churn with realistic patterns
        churn_prob = np.zeros(n_samples)
        churn_prob += (self.df['Tenure_Months'] < 10) * 0.4
        churn_prob += (self.df['Contract_Type'] == 'Month-to-Month') * 0.35
        churn_prob += (self.df['Internet_Service'] == 'Fiber') * 0.15
        churn_prob += (self.df['Tech_Support'] == 'No') * 0.2
        churn_prob = np.clip(churn_prob, 0, 1)

        self.df['Churn'] = (np.random.random(n_samples) < churn_prob).astype(int)

        print(f"✓ Dataset generated: {self.df.shape}")
        print(f"  Churn rate: {self.df['Churn'].mean():.2%}")
        return self.df

    def preprocess(self):
        """Preprocess data and split into train/test"""
        print("\nPreprocessing data...")

        df_processed = self.df.copy()

        # Encode categorical variables
        categorical_cols = ['Internet_Service', 'Contract_Type', 'Tech_Support', 'Online_Security']
        for col in categorical_cols:
            le = LabelEncoder()
            df_processed[col] = le.fit_transform(df_processed[col])

        # Separate features and target
        X = df_processed.drop(['CustomerID', 'Churn'], axis=1)
        y = df_processed['Churn']

        # Split data with stratification
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state, stratify=y
        )

        # Scale features
        scaler = StandardScaler()
        self.X_train_scaled = scaler.fit_transform(self.X_train)
        self.X_test_scaled = scaler.transform(self.X_test)

        print(f"✓ Training set: {self.X_train_scaled.shape}")
        print(f"✓ Testing set: {self.X_test_scaled.shape}")

        self.feature_names = X.columns

    def train_models(self):
        """Train Logistic Regression and Decision Tree"""
        print("\nTraining models...")

        # Logistic Regression
        self.lr_model = LogisticRegression(random_state=self.random_state, max_iter=1000)
        self.lr_model.fit(self.X_train_scaled, self.y_train)
        print("✓ Logistic Regression trained")

        # Decision Tree
        self.dt_model = DecisionTreeClassifier(max_depth=5, random_state=self.random_state, min_samples_split=10)
        self.dt_model.fit(self.X_train_scaled, self.y_train)
        print("✓ Decision Tree trained")

    def evaluate_models(self):
        """Evaluate both models"""
        print("\nEvaluating models...")

        # Predictions
        y_pred_lr = self.lr_model.predict(self.X_test_scaled)
        y_pred_proba_lr = self.lr_model.predict_proba(self.X_test_scaled)[:, 1]

        y_pred_dt = self.dt_model.predict(self.X_test_scaled)
        y_pred_proba_dt = self.dt_model.predict_proba(self.X_test_scaled)[:, 1]

        # Calculate metrics
        self.metrics_lr = {
            'y_pred': y_pred_lr,
            'y_pred_proba': y_pred_proba_lr,
            'Accuracy': accuracy_score(self.y_test, y_pred_lr),
            'Precision': precision_score(self.y_test, y_pred_lr),
            'Recall': recall_score(self.y_test, y_pred_lr),
            'F1-Score': f1_score(self.y_test, y_pred_lr),
            'ROC-AUC': roc_auc_score(self.y_test, y_pred_proba_lr),
        }

        self.metrics_dt = {
            'y_pred': y_pred_dt,
            'y_pred_proba': y_pred_proba_dt,
            'Accuracy': accuracy_score(self.y_test, y_pred_dt),
            'Precision': precision_score(self.y_test, y_pred_dt),
            'Recall': recall_score(self.y_test, y_pred_dt),
            'F1-Score': f1_score(self.y_test, y_pred_dt),
            'ROC-AUC': roc_auc_score(self.y_test, y_pred_proba_dt),
        }

        print("✓ Metrics calculated")

    def print_results(self):
        """Print detailed results"""
        print("\n" + "="*70)
        print("LOGISTIC REGRESSION METRICS")
        print("="*70)
        for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
            print(f"{metric:.<20} {self.metrics_lr[metric]:.4f}")

        print("\n" + "="*70)
        print("DECISION TREE METRICS")
        print("="*70)
        for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
            print(f"{metric:.<20} {self.metrics_dt[metric]:.4f}")

        # Classification reports
        print("\n" + "="*70)
        print("CLASSIFICATION REPORT - LOGISTIC REGRESSION")
        print("="*70)
        print(classification_report(self.y_test, self.metrics_lr['y_pred'], target_names=['No Churn', 'Churn']))

        print("="*70)
        print("CLASSIFICATION REPORT - DECISION TREE")
        print("="*70)
        print(classification_report(self.y_test, self.metrics_dt['y_pred'], target_names=['No Churn', 'Churn']))

    def visualize_confusion_matrices(self):
        """Create 4-panel confusion matrix dashboard"""
        print("\nGenerating confusion matrix dashboard...")

        cm_lr = confusion_matrix(self.y_test, self.metrics_lr['y_pred'])
        cm_dt = confusion_matrix(self.y_test, self.metrics_dt['y_pred'])

        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        fig.suptitle('Confusion Matrix Dashboard - Customer Churn Classification', fontsize=16, fontweight='bold')

        # LR - Counts
        sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
                   xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'],
                   cbar_kws={'label': 'Count'})
        axes[0, 0].set_title('Logistic Regression - Counts', fontweight='bold')
        axes[0, 0].set_ylabel('Actual')
        axes[0, 0].set_xlabel('Predicted')

        # LR - Normalized
        cm_lr_norm = cm_lr.astype('float') / cm_lr.sum(axis=1)[:, np.newaxis]
        sns.heatmap(cm_lr_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[0, 1],
                   xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'],
                   cbar_kws={'label': 'Percentage'})
        axes[0, 1].set_title('Logistic Regression - Normalized', fontweight='bold')
        axes[0, 1].set_ylabel('Actual')
        axes[0, 1].set_xlabel('Predicted')

        # DT - Counts
        sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[1, 0],
                   xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'],
                   cbar_kws={'label': 'Count'})
        axes[1, 0].set_title('Decision Tree - Counts', fontweight='bold')
        axes[1, 0].set_ylabel('Actual')
        axes[1, 0].set_xlabel('Predicted')

        # DT - Normalized
        cm_dt_norm = cm_dt.astype('float') / cm_dt.sum(axis=1)[:, np.newaxis]
        sns.heatmap(cm_dt_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1, 1],
                   xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'],
                   cbar_kws={'label': 'Percentage'})
        axes[1, 1].set_title('Decision Tree - Normalized', fontweight='bold')
        axes[1, 1].set_ylabel('Actual')
        axes[1, 1].set_xlabel('Predicted')

        plt.tight_layout()
        plt.savefig('confusion_matrix_dashboard.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: confusion_matrix_dashboard.png")
        plt.close()

    def visualize_pr_curves(self):
        """Create precision-recall curves"""
        print("Generating precision-recall curves...")

        precision_lr, recall_lr, _ = precision_recall_curve(self.y_test, self.metrics_lr['y_pred_proba'])
        precision_dt, recall_dt, _ = precision_recall_curve(self.y_test, self.metrics_dt['y_pred_proba'])

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('Precision-Recall Curves', fontsize=14, fontweight='bold')

        # LR PR Curve
        axes[0].plot(recall_lr, precision_lr, marker='o', linewidth=2, markersize=4, label='Logistic Regression', color='#1f77b4')
        axes[0].fill_between(recall_lr, precision_lr, alpha=0.2, color='#1f77b4')
        axes[0].set_xlabel('Recall', fontsize=11)
        axes[0].set_ylabel('Precision', fontsize=11)
        axes[0].set_title('Logistic Regression', fontweight='bold')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend(loc='best')
        axes[0].set_xlim([-0.05, 1.05])
        axes[0].set_ylim([-0.05, 1.05])

        # DT PR Curve
        axes[1].plot(recall_dt, precision_dt, marker='s', linewidth=2, markersize=4, label='Decision Tree', color='#2ca02c')
        axes[1].fill_between(recall_dt, precision_dt, alpha=0.2, color='#2ca02c')
        axes[1].set_xlabel('Recall', fontsize=11)
        axes[1].set_ylabel('Precision', fontsize=11)
        axes[1].set_title('Decision Tree', fontweight='bold')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend(loc='best')
        axes[1].set_xlim([-0.05, 1.05])
        axes[1].set_ylim([-0.05, 1.05])

        plt.tight_layout()
        plt.savefig('precision_recall_curves.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: precision_recall_curves.png")
        plt.close()

    def visualize_roc_curves(self):
        """Create ROC curves"""
        print("Generating ROC curves...")

        fpr_lr, tpr_lr, _ = roc_curve(self.y_test, self.metrics_lr['y_pred_proba'])
        roc_auc_lr = auc(fpr_lr, tpr_lr)

        fpr_dt, tpr_dt, _ = roc_curve(self.y_test, self.metrics_dt['y_pred_proba'])
        roc_auc_dt = auc(fpr_dt, tpr_dt)

        plt.figure(figsize=(10, 8))
        plt.plot(fpr_lr, tpr_lr, linewidth=2.5, label=f'Logistic Regression (AUC = {roc_auc_lr:.4f})', color='#1f77b4')
        plt.plot(fpr_dt, tpr_dt, linewidth=2.5, label=f'Decision Tree (AUC = {roc_auc_dt:.4f})', color='#2ca02c')
        plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier', alpha=0.7)

        plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
        plt.legend(loc='lower right', fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: roc_curves.png")
        plt.close()

    def visualize_model_comparison(self):
        """Create model comparison visualization"""
        print("Generating model comparison...")

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Metrics comparison
        metrics_comparison = pd.DataFrame({
            'Logistic Regression': self.metrics_lr,
            'Decision Tree': self.metrics_dt
        }).drop(['y_pred', 'y_pred_proba'])

        metrics_comparison.T.plot(kind='bar', ax=axes[0], width=0.8)
        axes[0].set_title('Model Performance Comparison', fontweight='bold', fontsize=12)
        axes[0].set_ylabel('Score', fontweight='bold')
        axes[0].set_xlabel('Model', fontweight='bold')
        axes[0].set_ylim([0, 1])
        axes[0].legend(loc='lower right', fontsize=9)
        axes[0].grid(True, alpha=0.3, axis='y')
        plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=0)

        # Feature importance
        feature_importance = pd.DataFrame({
            'Feature': self.feature_names,
            'Importance': self.dt_model.feature_importances_
        }).sort_values('Importance', ascending=True)

        feature_importance.plot(x='Feature', y='Importance', kind='barh', ax=axes[1], legend=False, color='#2ca02c')
        axes[1].set_title('Feature Importance - Decision Tree', fontweight='bold', fontsize=12)
        axes[1].set_xlabel('Importance Score', fontweight='bold')
        axes[1].grid(True, alpha=0.3, axis='x')

        plt.tight_layout()
        plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
        print("✓ Saved: model_comparison.png")
        plt.close()

    def generate_report(self):
        """Generate comprehensive report"""
        print("Generating report...")

        cm_lr = confusion_matrix(self.y_test, self.metrics_lr['y_pred'])
        cm_dt = confusion_matrix(self.y_test, self.metrics_dt['y_pred'])

        feature_importance = pd.DataFrame({
            'Feature': self.feature_names,
            'Importance': self.dt_model.feature_importances_
        }).sort_values('Importance', ascending=False)

        report = f"""
{'='*70}
CUSTOMER CHURN PREDICTION - FINAL REPORT
{'='*70}

DATASET SUMMARY
{'-'*70}
Total Samples: {len(self.df)}
Training Samples: {len(self.X_train)}
Testing Samples: {len(self.X_test)}
Churn Rate: {self.df['Churn'].mean():.2%}
Number of Features: {len(self.feature_names)}

MODEL PERFORMANCE - LOGISTIC REGRESSION
{'-'*70}
Accuracy:  {self.metrics_lr['Accuracy']:.4f}
Precision: {self.metrics_lr['Precision']:.4f}
Recall:    {self.metrics_lr['Recall']:.4f}
F1-Score:  {self.metrics_lr['F1-Score']:.4f}
ROC-AUC:   {self.metrics_lr['ROC-AUC']:.4f}

CONFUSION MATRIX - LOGISTIC REGRESSION
{'-'*70}
True Negatives:  {cm_lr[0,0]} | False Positives: {cm_lr[0,1]}
False Negatives: {cm_lr[1,0]} | True Positives:  {cm_lr[1,1]}

MODEL PERFORMANCE - DECISION TREE
{'-'*70}
Accuracy:  {self.metrics_dt['Accuracy']:.4f}
Precision: {self.metrics_dt['Precision']:.4f}
Recall:    {self.metrics_dt['Recall']:.4f}
F1-Score:  {self.metrics_dt['F1-Score']:.4f}
ROC-AUC:   {self.metrics_dt['ROC-AUC']:.4f}

CONFUSION MATRIX - DECISION TREE
{'-'*70}
True Negatives:  {cm_dt[0,0]} | False Positives: {cm_dt[0,1]}
False Negatives: {cm_dt[1,0]} | True Positives:  {cm_dt[1,1]}

TOP 3 IMPORTANT FEATURES (Decision Tree)
{'-'*70}
"""
        for idx, row in feature_importance.head(3).iterrows():
            report += f"{row['Feature']:.<25} {row['Importance']:.4f}\n"

        report += f"""
VISUALIZATIONS GENERATED
{'-'*70}
✓ confusion_matrix_dashboard.png
✓ precision_recall_curves.png
✓ roc_curves.png
✓ model_comparison.png

{'='*70}
"""

        with open('churn_classification_report.txt', 'w') as f:
            f.write(report)

        print("✓ Saved: churn_classification_report.txt")

    def run(self):
        """Run complete pipeline"""
        print("\n" + "="*70)
        print("CUSTOMER CHURN CLASSIFICATION PIPELINE")
        print("="*70)

        self.generate_dataset()
        self.preprocess()
        self.train_models()
        self.evaluate_models()
        self.print_results()
        self.visualize_confusion_matrices()
        self.visualize_pr_curves()
        self.visualize_roc_curves()
        self.visualize_model_comparison()
        self.generate_report()

        print("\n" + "="*70)
        print("✓ PIPELINE COMPLETE")
        print("="*70)
        print("\nOutput files generated:")
        print("  1. confusion_matrix_dashboard.png")
        print("  2. precision_recall_curves.png")
        print("  3. roc_curves.png")
        print("  4. model_comparison.png")
        print("  5. churn_classification_report.txt")
        print("\n")


if __name__ == "__main__":
    classifier = ChurnClassifier(random_state=42)
    classifier.run()


CUSTOMER CHURN CLASSIFICATION PIPELINE
Generating dataset...
✓ Dataset generated: (1000, 10)
  Churn rate: 36.20%

Preprocessing data...
✓ Training set: (800, 8)
✓ Testing set: (200, 8)

Training models...
✓ Logistic Regression trained
✓ Decision Tree trained

Evaluating models...
✓ Metrics calculated

LOGISTIC REGRESSION METRICS
Accuracy............ 0.7750
Precision........... 0.7755
Recall.............. 0.5278
F1-Score............ 0.6281
ROC-AUC............. 0.8127

DECISION TREE METRICS
Accuracy............ 0.7800
Precision........... 0.7258
Recall.............. 0.6250
F1-Score............ 0.6716
ROC-AUC............. 0.7696

CLASSIFICATION REPORT - LOGISTIC REGRESSION
              precision    recall  f1-score   support

    No Churn       0.77      0.91      0.84       128
       Churn       0.78      0.53      0.63        72

    accuracy                           0.78       200
   macro avg       0.78      0.72      0.73       200
weighted avg       0.78      0.78      0.76    